# Fine-tune **ArcFace-style** recognition on your gallery (PyTorch)

This notebook **does not** call DeepFace for training. It uses **PyTorch** with:

- A **ResNet-50** backbone pretrained on **ImageNet** (via `torchvision`) — a practical starting point on a laptop.
- An **ArcFace (additive angular margin)** classification head (`ArcMarginProduct`) so training matches the *idea* of ArcFace (angular margin between identities).

**Honest scope:** “Full” ArcFace in papers often starts from face-specific pretraining on millions of faces. Here you **adapt** a strong CNN to **your** `photos_all_faces` IDs with limited data — useful for course experiments and domain shift, not a guarantee to beat InsightFace’s released checkpoints.

**Hardware:** On Apple Silicon, this notebook uses **`mps`** when available (`M5 Max`, unified memory). Lower `BATCH_SIZE` if you hit memory limits.

**Data:** Filenames like `a_gp_....jpg` → identity `A`. Training uses **`open_data_set/photos_all_faces_cropped/`** when that folder has images, otherwise **`photos_all_faces`**. Set **`MERGE_BOTH_GALLERIES = True`** in section 1 to train on the union of both folders (deduped by file path). All images in the chosen folder(s) are used — no per-ID cap.


## 0) Dependencies

Run once in your **project venv** (same as other notebooks). If PyTorch is missing, uncomment the install line.


In [ ]:
# !pip install torch torchvision tqdm scikit-learn pillow pandas

import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)



## 1) Discover gallery images + identity labels

Same convention as `people_identification_in_the_wild_*.ipynb`: first filename token → identity letter. By default, **cropped** images are used for training when `photos_all_faces_cropped` exists.


In [ ]:
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"
PHOTOS_ALL_FACES = DATASET_ROOT / "photos_all_faces"
PHOTOS_CROPPED = DATASET_ROOT / "photos_all_faces_cropped"

# True = train on every image from BOTH folders (paths deduped). False = prefer cropped only, else photos_all_faces.
MERGE_BOTH_GALLERIES = False

def list_images(folder: Path):
    out = []
    if not folder.exists():
        return out
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        out.extend(folder.glob(ext))
    return sorted(out)

def infer_identity(path: Path) -> str:
    return path.stem.split("_")[0].upper()

rows = []
seen_paths = set()
if MERGE_BOTH_GALLERIES:
    folders = [PHOTOS_CROPPED, PHOTOS_ALL_FACES]
    for folder in folders:
        for p in list_images(folder):
            key = str(p.resolve())
            if key in seen_paths:
                continue
            seen_paths.add(key)
            rows.append({"path": key, "identity": infer_identity(p)})
    GALLERY_DIR = PHOTOS_CROPPED if PHOTOS_CROPPED.is_dir() else PHOTOS_ALL_FACES
    print("MERGE_BOTH_GALLERIES: using images from:", [str(f) for f in folders if f.is_dir()])
else:
    if PHOTOS_CROPPED.is_dir() and list_images(PHOTOS_CROPPED):
        GALLERY_DIR = PHOTOS_CROPPED
    elif PHOTOS_ALL_FACES.is_dir() and list_images(PHOTOS_ALL_FACES):
        GALLERY_DIR = PHOTOS_ALL_FACES
    else:
        raise FileNotFoundError(
            f"No images found. Add files under {PHOTOS_CROPPED} or {PHOTOS_ALL_FACES}."
        )
    print("Training gallery:", GALLERY_DIR.resolve())
    for p in list_images(GALLERY_DIR):
        rows.append({"path": str(p.resolve()), "identity": infer_identity(p)})

gallery_df = pd.DataFrame(rows)
if len(gallery_df) == 0:
    raise FileNotFoundError("No gallery rows built. Check MERGE_BOTH_GALLERIES and dataset folders.")

# Drop identities with too few images for a stratified val split
MIN_IMAGES_PER_ID = 2
cnt = gallery_df["identity"].value_counts()
keep_ids = cnt[cnt >= MIN_IMAGES_PER_ID].index
gallery_df = gallery_df[gallery_df["identity"].isin(keep_ids)].reset_index(drop=True)
if len(gallery_df) == 0:
    raise RuntimeError(f"No identity has >= {MIN_IMAGES_PER_ID} images after filtering.")

identities = sorted(gallery_df["identity"].unique())
label_to_idx = {lab: i for i, lab in enumerate(identities)}
num_classes = len(identities)
print(f"Gallery images: {len(gallery_df)} | identities: {num_classes}")
display(gallery_df["identity"].value_counts().to_frame("n_images"))



## 2) Train / validation split

Stratified by identity so each ID appears in both splits (requires `MIN_IMAGES_PER_ID` ≥ 2).


In [ ]:
from sklearn.model_selection import train_test_split

VAL_FRAC = 0.15
train_df, val_df = train_test_split(
    gallery_df,
    test_size=VAL_FRAC,
    stratify=gallery_df["identity"],
    random_state=SEED,
)
print("Train:", len(train_df), " Val:", len(val_df))



## 3) Dataset + transforms

Resize + flip + tensor normalize (ImageNet mean/std — standard when starting from ImageNet weights).


In [ ]:
IMG_SIZE = 112  # common for face pipelines; small = faster on laptop

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class FaceIdDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tfm):
        self.paths = frame["path"].tolist()
        self.labels = [label_to_idx[i] for i in frame["identity"].tolist()]
        self.tfm = tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        x = self.tfm(img)
        y = self.labels[idx]
        return x, y


train_ds = FaceIdDataset(train_df, train_tfms)
val_ds = FaceIdDataset(val_df, val_tfms)



## 4) ArcMargin product (ArcFace-style logits)

Implements the additive angular margin loss interface: forward takes **normalized embeddings** and **class indices**, returns scaled logits for `CrossEntropyLoss`.


In [ ]:
class ArcMarginProduct(nn.Module):
    """ArcFace margin on cosine similarity (simplified standard implementation)."""
    def __init__(self, in_features: int, out_features: int, s: float = 30.0, m: float = 0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        emb = F.normalize(emb, dim=1)
        W = F.normalize(self.weight, dim=1)
        cosine = F.linear(emb, W)
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=0.0, max=1.0))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s


class FaceNet(nn.Module):
    """ResNet50 → embedding dim → ArcMargin logits during training."""
    def __init__(self, embedding_dim: int = 512, num_classes: int = 11):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # drop FC
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.bn = nn.BatchNorm1d(2048)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(2048, embedding_dim)
        self.arc = ArcMarginProduct(embedding_dim, num_classes, s=30.0, m=0.35)

    def forward(self, x: torch.Tensor, labels=None):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.dropout(x)
        emb = self.fc(x)
        if labels is None:
            return F.normalize(emb, dim=1)
        return self.arc(emb, labels)



## 5) Hyperparameters + training

Defaults target a **stronger** run on Apple Silicon (e.g. M5 Max): more epochs and **`BATCH_SIZE = 32`**. If MPS OOMs, try **24** or **16**. Training uses **`drop_last=False`** so every image is seen each epoch (a tiny last batch is normal).


In [ ]:
# ----- knobs -----
EPOCHS = 40
BATCH_SIZE = 32  # try 48 on 36GB unified mem; lower to 16 if MPS OOM
LR = 1e-4
WEIGHT_DECAY = 1e-4
EMBEDDING_DIM = 512
# -----------------

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

model = FaceNet(embedding_dim=EMBEDDING_DIM, num_classes=num_classes).to(DEVICE)
# Train backbone + head (full fine-tune on your data). For less overfit, freeze early layers:
# for p in list(model.features.parameters())[:-20]:
#     p.requires_grad = False

opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS, 1))
ce = nn.CrossEntropyLoss()


@torch.no_grad()
def accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb, yb)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.numel()
    return correct / max(total, 1)


best_val = 0.0
best_path = PROJECT_ROOT / "finetune_arcface_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(xb, yb)
        loss = ce(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        losses.append(loss.item())
    sched.step()
    tr_loss = float(np.mean(losses)) if losses else 0.0
    va = accuracy(val_loader)
    print(f"Epoch {epoch}: train_loss={tr_loss:.4f}  val_acc={va:.4f}")
    if va >= best_val:
        best_val = va
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "label_to_idx": label_to_idx,
                "identities": identities,
                "embedding_dim": EMBEDDING_DIM,
                "img_size": IMG_SIZE,
                "train_gallery": (
                    "merged:"
                    + ",".join(
                        str(f.resolve())
                        for f in (PHOTOS_CROPPED, PHOTOS_ALL_FACES)
                        if f.is_dir()
                    )
                    if MERGE_BOTH_GALLERIES
                    else str(GALLERY_DIR.resolve())
                ),
                "merge_galleries": MERGE_BOTH_GALLERIES,
            },
            best_path,
        )
        print("  saved ->", best_path)

print("Done. Best val acc:", round(best_val, 4))



## 6) Embedding checks

Uses the **same image index as section 1** (`gallery_df`) so tests match what you trained on (including **`MERGE_BOTH_GALLERIES`**). **Tests:** same-identity vs different-identity cosine means; tune **`RELATIVE_MARGIN`** / **`SAME_ID_MIN_MEAN`** if needed.


In [ ]:
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("Embedding tests on gallery_df:", len(gallery_df), "images | source:", ckpt.get("train_gallery", "?"), "| merged:", ckpt.get("merge_galleries", "?"))


def embed_path(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    x = val_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        e = model(x, labels=None).cpu().numpy()[0]
    return e


def cos_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


test_df = gallery_df.copy()
if len(test_df) == 0:
    raise FileNotFoundError("gallery_df is empty; rerun section 1.")

tcnt = test_df["identity"].value_counts()
test_df = test_df[test_df["identity"].isin(tcnt[tcnt >= 2].index)].reset_index(drop=True)
if len(test_df) == 0:
    raise RuntimeError("Need at least one identity with 2+ images in the test folder for same-ID checks.")

# --- test hyperparameters (tune if asserts fail) ---
RELATIVE_MARGIN = 0.05   # mean(same) must exceed mean(diff) by this much
SAME_ID_MIN_MEAN = 0.15  # floor on mean same-ID cosine (set lower for checkpoints right after 1 epoch)
N_SAME_PAIRS = len(test_df["identity"].unique())  # one same-ID pair per identity (all IDs)
N_DIFF_PAIRS = 128
rng = np.random.default_rng(SEED)

# Same-identity pairs: two distinct images per identity (first two paths in sorted order for reproducibility)
same_sims = []
examples_same = []
for uid in sorted(test_df["identity"].unique())[:N_SAME_PAIRS]:
    sub = test_df[test_df["identity"] == uid].sort_values("path")
    if len(sub) < 2:
        continue
    p0, p1 = sub.iloc[0]["path"], sub.iloc[1]["path"]
    s = cos_sim(embed_path(p0), embed_path(p1))
    same_sims.append(s)
    examples_same.append((uid, round(s, 4)))

# Different-identity pairs: random A,B with A != B
ids_all = test_df["identity"].unique().tolist()
diff_sims = []
if len(ids_all) < 2:
    print("Note: only one identity in test folder — skipping different-ID pairs (need 2+ IDs for that check).")
else:
    for _ in range(N_DIFF_PAIRS):
        a, b = rng.choice(ids_all, size=2, replace=False)
        pa = rng.choice(test_df[test_df["identity"] == a]["path"].tolist())
        pb = rng.choice(test_df[test_df["identity"] == b]["path"].tolist())
        diff_sims.append(cos_sim(embed_path(pa), embed_path(pb)))

mean_same = float(np.mean(same_sims)) if same_sims else 0.0
mean_diff = float(np.mean(diff_sims)) if diff_sims else float("nan")
print(f"Same-ID pairs: n={len(same_sims)}  mean_cos={mean_same:.4f}  min={min(same_sims):.4f}  max={max(same_sims):.4f}")
if diff_sims:
    print(f"Diff-ID pairs: n={len(diff_sims)}  mean_cos={mean_diff:.4f}  min={min(diff_sims):.4f}  max={max(diff_sims):.4f}")
print("Examples (identity, cos same two crops):", examples_same[:5])

# Assertions (test cases)
assert len(same_sims) > 0, "No same-identity pairs computed."
if len(ids_all) >= 2 and diff_sims:
    assert mean_same > mean_diff + RELATIVE_MARGIN, (
        f"Same-ID mean cosine {mean_same:.4f} should exceed diff-ID mean {mean_diff:.4f} by > {RELATIVE_MARGIN}. "
        "Train longer, tune ArcFace m/s, or lower RELATIVE_MARGIN for debugging."
    )
assert mean_same >= SAME_ID_MIN_MEAN, (
    f"Same-ID mean cosine {mean_same:.4f} is below SAME_ID_MIN_MEAN={SAME_ID_MIN_MEAN}. "
    "Lower SAME_ID_MIN_MEAN if the model is still weak."
)
print("OK: embedding tests passed.")



## Next steps (optional)

- **Use embeddings in your pipeline:** export all gallery embeddings with `model(x, labels=None)` and replace DeepFace’s `represent` in matching code (same cosine threshold ideas).
- **Stronger starting weights:** plug in a face-pretrained backbone checkpoint (e.g. InsightFace zoo) instead of ImageNet-only ResNet50 — same ArcFace head, better starting point.
- **Regularization:** more data per ID, stronger augmentations, or freeze early ResNet layers if you overfit.
